In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 105.3 MB/s eta 0:00:00


In [ ]:
!pip install faiss-cpu # Installing faiss-cpu, which works on Linux machines without a GPU. If you have a compatible GPU and CUDA installed, you can try installing 'faiss-gpu' for better performance.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 80.2 MB/s eta 0:00:00


In [ ]:
!pip install pdf2image

In [ ]:
!pip install pytesseract

In [ ]:
import os
import re
import io
import numpy as np
import pandas as pd
import pdfplumber
from PIL import Image
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer, util
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer as ViTAutoTokenizer
from pdf2image import convert_from_path
import warnings
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import cv2
import pytesseract

In [ ]:
import os
import re
import io
import numpy as np
import pandas as pd
import pdfplumber
from PIL import Image
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer, util
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer as ViTAutoTokenizer
from pdf2image import convert_from_path
import warnings
import string
from sklearn.feature_extraction.text import TfidfVectorizer
import cv2
import pytesseract

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
BASE_DIR = '/content/'  # Adjust as needed
DOCUMENT_FILE = '/content/drive/MyDrive/lab2/document.pdf'  # Your provided path
QUESTIONS_FILE = os.path.join(BASE_DIR, '/content/drive/MyDrive/lab2/Lab_2_Part_1_Questions.csv')  # Adjust if needed
FIGURE_SAVE_DIR = 'data/figures'
CACHE_DIR = 'data/cache'

# Create directories
os.makedirs(FIGURE_SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Model configuration
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"  # Smaller model for speed
TEXT_EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
GENERATOR_MODEL_NAME = "google/flan-t5-large"
IMAGE_CAPTION_MODEL_NAME = "nlpconnect/vit-gpt2-image-captioning"

In [ ]:
def init_reranker(model_name=RERANKER_MODEL_NAME):
    """Initialize the reranker model"""
    print(f"Loading reranker model: {model_name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        print(f"Reranker model loaded on {device}")
        return model, tokenizer, device
    except Exception as e:
        print(f"Error loading reranker model: {e}")
        return None, None, None

def init_generator(model_name=GENERATOR_MODEL_NAME):
    """Initialize text generation model"""
    print(f"Loading text generation model: {model_name}...")
    try:
        device = 0 if torch.cuda.is_available() else -1
        generator = pipeline('text2text-generation', model=model_name, max_length=250, device=device)
        print(f"Generator model loaded on device {device}")
        return generator
    except Exception as e:
        print(f"Error loading generator model: {e}")
        return None

def init_image_caption_model(model_name=IMAGE_CAPTION_MODEL_NAME):
    """Initialize image captioning model"""
    print(f"Loading image captioning model: {model_name}...")
    try:
        model = VisionEncoderDecoderModel.from_pretrained(model_name)
        feature_extractor = ViTImageProcessor.from_pretrained(model_name)
        tokenizer = ViTAutoTokenizer.from_pretrained(model_name)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval()
        print(f"Image captioning model loaded on {device}")
        return model, feature_extractor, tokenizer, device
    except Exception as e:
        print(f"Error loading image captioning model: {e}")
        return None, None, None, None

In [ ]:
def generate_image_caption(image_data, model, feature_extractor, tokenizer, device):
    """Generate caption for an image"""
    if model is None or feature_extractor is None or tokenizer is None or image_data is None:
        return ""
    try:
        image = Image.open(io.BytesIO(image_data)).convert("RGB")
        pixel_values = feature_extractor(images=[image], return_tensors="pt").pixel_values.to(device)

        with torch.no_grad():
            output_ids = model.generate(pixel_values, max_length=64, num_beams=4, early_stopping=True)

        caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        return caption.strip()
    except Exception as e:
        print(f"Error generating image caption: {e}")
        return ""

def extract_text_from_image(image_data):
    """Extract text from image using OCR"""
    if image_data is None:
        return ""
    try:
        image = Image.open(io.BytesIO(image_data))
        # Convert to grayscale for better OCR
        opencvImage = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
        # Apply thresholding
        thresh = cv2.threshold(opencvImage, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

        # Tesseract configuration
        custom_config = r'--oem 3 --psm 6'
        text = pytesseract.image_to_string(thresh, config=custom_config)
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from image: {e}")
        return ""

In [ ]:
def rerank(query, passages, model, tokenizer, device, top_k=8, batch_size=8):
    """Rerank passages using the reranker model with better truncation"""
    if not passages or model is None:
        return passages[:top_k] if passages else []

    paired_texts = []
    original_indices = []

    for i, passage in enumerate(passages):
        text = passage['text']
        # More aggressive truncation - use only first 300 words
        text_words = text.split()
        if len(text_words) > 300:
            text = ' '.join(text_words[:300])
        paired_texts.append([query, text])
        original_indices.append(i)

    scores = []
    with torch.no_grad():
        for i in range(0, len(paired_texts), batch_size):
            batch = paired_texts[i:i+batch_size]
            try:
                inputs = tokenizer(
                    batch, padding=True, truncation=True,
                    return_tensors="pt", max_length=384  # Reduced from 512
                ).to(device)
                batch_scores = model(**inputs).logits.squeeze(-1)
                if batch_scores.ndim == 0:
                    batch_scores = batch_scores.unsqueeze(0)
                scores.extend(batch_scores.cpu().tolist())
            except Exception as e:
                print(f"Error during batch processing: {e}")
                # Assign default scores if batch fails
                scores.extend([0.0] * len(batch))

    # Assign scores
    scored_passages = []
    for i, score in enumerate(scores):
        idx = original_indices[i]
        if idx < len(passages):
            passage_copy = passages[idx].copy()
            passage_copy['rerank_score'] = float(score)
            scored_passages.append(passage_copy)

    # Sort by score
    reranked_passages = sorted(scored_passages, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return reranked_passages[:top_k]

In [ ]:
def extract_text_and_figures(pdf_path, caption_model, feature_extractor, tokenizer, device, dpi=100):
    """Extract text, tables, and figures from PDF"""
    text_chunks = []
    figures_tables = []  # Combined list
    mention_pattern = re.compile(r'(figure|table)\s+(\d+-\d+)', re.IGNORECASE)

    # Pass 1: Extract text and find mentions
    print("Extracting text and finding figure/table mentions...")
    page_texts = {}
    page_mentions = {}
    page_tables = {}

    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for page_num, page in enumerate(tqdm(pdf.pages, desc="Parsing PDF")):
                # Extract text
                page_text = page.extract_text(x_tolerance=1, y_tolerance=1)
                if page_text:
                    page_texts[page_num] = page_text
                    text_chunks.append({'page_num': page_num, 'text': page_text.strip()})

                    # Find figure/table mentions
                    mentions = []
                    for match in mention_pattern.finditer(page_text):
                        fig_type = match.group(1).lower()
                        fig_num_str = match.group(2)
                        start = max(0, match.start() - 150)
                        end = min(len(page_text), match.end() + 150)
                        context = page_text[start:end].strip().replace('\n', ' ')
                        mentions.append((fig_type, fig_num_str, context))
                    if mentions:
                        page_mentions[page_num] = mentions

                # Extract tables
                tables = page.extract_tables()
                if tables:
                    table_texts = []
                    for table_data in tables:
                        if table_data:
                            table_text = "\n".join([" | ".join([str(cell or "").replace('\n', ' ').strip() for cell in row]) for row in table_data])
                            table_texts.append(table_text)
                    if table_texts:
                        page_tables[page_num] = table_texts
    except Exception as e:
        print(f"Error during PDF text extraction: {e}")
        return [], []

    print(f"Found mentions on {len(page_mentions)} pages.")
    print(f"Found tables on {len(page_tables)} pages.")

    # Pass 2: Generate page images and process figures/tables
    print("Generating page images and processing figures/tables...")
    pil_page_images = []
    try:
        pil_page_images = convert_from_path(pdf_path, dpi=dpi)
        if len(pil_page_images) != total_pages:
            print(f"Warning: Page count mismatch! pdfplumber={total_pages}, pdf2image={len(pil_page_images)}")
    except Exception as e:
        print(f"Error converting PDF to images: {e}")

    # Create entries for each figure/table mention
    processed_mentions = set()
    for page_num, mentions in page_mentions.items():
        for fig_type, fig_num_str, context in mentions:
            mention_key = (page_num, fig_type, fig_num_str)
            if mention_key not in processed_mentions:
                figures_tables.append({
                    'type': fig_type,
                    'number_str': fig_num_str,
                    'page_num': page_num,
                    'caption_context': context,
                    'image_data': None,
                    'ocr_text': "",
                    'generated_caption': "",
                    'table_text': "",
                    'description_embedding': None,
                    'detected_by': 'text_mention',
                })
                processed_mentions.add(mention_key)

    # Process figures/tables
    print("Processing figures/tables...")
    for i, item in enumerate(tqdm(figures_tables, desc="Processing Items")):
        page_num = item['page_num']
        item_type = item['type']
        item_num_str = item['number_str']

        # Assign image data
        if page_num < len(pil_page_images):
            try:
                img_byte_arr = io.BytesIO()
                pil_page_images[page_num].save(img_byte_arr, format='PNG')
                item['image_data'] = img_byte_arr.getvalue()

                # Run OCR and captioning
                item['ocr_text'] = extract_text_from_image(item['image_data'])
                item['generated_caption'] = generate_image_caption(
                    item['image_data'], caption_model, feature_extractor, tokenizer, device
                )

                # Save image
                img_save_path = os.path.join(FIGURE_SAVE_DIR, f"{item_type}_{item_num_str}_p{page_num}.png")
                with open(img_save_path, 'wb') as f_img:
                    f_img.write(item['image_data'])
            except Exception as e:
                print(f"Error processing image: {e}")
                item['image_data'] = None

        # Assign table text if available
        if item_type == 'table' and page_num in page_tables and page_tables[page_num]:
            item['table_text'] = page_tables[page_num].pop(0)
            item['detected_by'] += '+plumber_table'
            if not item['ocr_text'] and item['table_text']:
                item['ocr_text'] = item['table_text'][:1000]

    # Add remaining tables found by pdfplumber but not mentioned
    for page_num, remaining_tables in page_tables.items():
        for table_text in remaining_tables:
            table_num = max([int(f['number_str'].split('-')[1]) for f in figures_tables if f['type']=='table' and '-' in f['number_str']], default=0) + 1
            chapter_num = min([int(f['number_str'].split('-')[0]) for f in figures_tables if f['type']=='table' and '-' in f['number_str']], default=1)
            new_num_str = f"{chapter_num}-{table_num}"
            print(f"Adding unmentioned table on page {page_num}, assigning number {new_num_str}")
            figures_tables.append({
                'type': 'table',
                'number_str': new_num_str,
                'page_num': page_num,
                'caption_context': f"Table detected on page {page_num+1}",
                'image_data': None,
                'ocr_text': table_text[:1000],
                'generated_caption': "",
                'table_text': table_text,
                'description_embedding': None,
                'detected_by': 'plumber_table_unmentioned',
            })

    # Create description embeddings
    print("Creating description embeddings...")
    text_embed_model = None
    try:
        text_embed_model = SentenceTransformer(TEXT_EMBEDDING_MODEL_NAME)

        for item in tqdm(figures_tables, desc="Creating Embeddings"):
            desc_parts = []
            if item.get('caption_context'):
                desc_parts.append(item['caption_context'])
            if item.get('generated_caption'):
                desc_parts.append(item['generated_caption'])
            if item['type'] == 'table' and item.get('table_text'):
                desc_parts.append(item['table_text'][:1000])
            elif item.get('ocr_text'):
                desc_parts.append(item['ocr_text'][:1000])

            if desc_parts:
                full_description = " ".join(desc_parts)
                full_description = re.sub(r'\s+', ' ', full_description).strip()
                if full_description:
                    item['description_embedding'] = text_embed_model.encode(full_description)

        embed_count = sum(1 for f in figures_tables if f.get('description_embedding') is not None)
        print(f"Created embeddings for {embed_count}/{len(figures_tables)} figures/tables")
    except Exception as e:
        print(f"Error creating embeddings: {e}")
        for item in figures_tables:
            item['description_embedding'] = None

    return text_chunks, figures_tables

In [ ]:
def preprocess_text(text):
    """Clean text for processing"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def process_text_chunks(text_chunks, figures_tables, chunk_size=500, overlap=100):
    """Process text into overlapping chunks with figure/table references"""
    processed_chunks = []
    mention_pattern = re.compile(r'(figure|table)\s+(\d+-\d+)', re.IGNORECASE)

    for chunk in text_chunks:
        page_num = chunk['page_num']
        page_text = chunk['text']
        words = page_text.split()

        start_idx = 0
        while start_idx < len(words):
            end_idx = min(start_idx + chunk_size, len(words))
            chunk_text = ' '.join(words[start_idx:end_idx])

            # Skip small chunks
            if len(chunk_text.split()) < 10:
                start_idx += (chunk_size - overlap)
                if start_idx >= len(words):
                    break
                continue

            # Find figure/table mentions
            figures_in_chunk = []
            chunk_lower = chunk_text.lower()
            for match in mention_pattern.finditer(chunk_lower):
                fig_type = match.group(1).lower()
                fig_num_str = match.group(2)
                figures_in_chunk.append(f"{fig_type} {fig_num_str}")

            processed_chunks.append({
                'page_num': page_num,
                'text': chunk_text,
                'start_word_idx': start_idx,
                'end_word_idx': end_idx,
                'figures_mentioned': list(set(figures_in_chunk)),
            })

            if end_idx == len(words):
                break
            start_idx += (chunk_size - overlap)
            if start_idx >= end_idx:
                start_idx = end_idx

    return processed_chunks

In [ ]:
def create_tfidf_vectorizer(chunks):
    """Create TF-IDF vectorizer and matrix"""
    texts = [chunk['text'] for chunk in chunks]
    preprocessed_texts = [preprocess_text(text) for text in texts]
    vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(preprocessed_texts)
    return vectorizer, tfidf_matrix

def create_text_embeddings(chunks, model_name=TEXT_EMBEDDING_MODEL_NAME):
    """Create embeddings for text chunks"""
    print(f"Creating text embeddings with {model_name}...")
    model = None
    try:
        model = SentenceTransformer(model_name)
    except Exception as e:
        print(f"Failed to load embedding model: {e}")
        for chunk in chunks:
            chunk['embedding'] = None
        return chunks, None

    texts = [chunk['text'] for chunk in chunks]
    try:
        embeddings = model.encode(texts, batch_size=16, show_progress_bar=True)
        for i, chunk in enumerate(chunks):
            chunk['embedding'] = embeddings[i]
        return chunks, model
    except Exception as e:
        print(f"Error encoding text: {e}")
        for chunk in chunks:
            chunk['embedding'] = None
        return chunks, model

def build_faiss_index(chunks_with_embeddings):
    """Build FAISS index for similarity search"""
    embeddings = [chunk['embedding'] for chunk in chunks_with_embeddings
                  if chunk.get('embedding') is not None and isinstance(chunk.get('embedding'), np.ndarray)]

    if not embeddings:
        print("No valid embeddings found for FAISS index.")
        return None

    embeddings_np = np.array(embeddings).astype('float32')
    if embeddings_np.ndim != 2 or embeddings_np.shape[0] == 0:
        print("Embeddings not in correct format for FAISS.")
        return None

    embedding_dim = embeddings_np.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)

    try:
        index.add(embeddings_np)
        print(f"FAISS index built with {index.ntotal} vectors.")
        return index
    except Exception as e:
        print(f"Error building FAISS index: {e}")
        return None

In [ ]:
def hybrid_retrieve(query, chunks, tfidf_vectorizer, tfidf_matrix, faiss_index, embedding_model, k=30, semantic_weight=0.7):
    """Hybrid retrieval combining TF-IDF and semantic search"""
    results = {}

    # Map chunks with valid embeddings to FAISS indices
    chunk_indices_with_embeddings = [i for i, chunk in enumerate(chunks)
                                      if chunk.get('embedding') is not None and
                                      isinstance(chunk.get('embedding'), np.ndarray)]

    if not chunk_indices_with_embeddings:
        print("No chunks with valid embeddings available.")
        semantic_weight = 0  # Fallback to TF-IDF only

    original_indices_map = {new_idx: old_idx for new_idx, old_idx in enumerate(chunk_indices_with_embeddings)}

    # Semantic search
    if faiss_index and embedding_model and semantic_weight > 0 and chunk_indices_with_embeddings:
        try:
            query_embedding = embedding_model.encode([query])[0].astype('float32').reshape(1, -1)
            limit = min(k, faiss_index.ntotal)

            if limit > 0:
                distances, faiss_indices = faiss_index.search(query_embedding, limit)
                for i, faiss_idx in enumerate(faiss_indices[0]):
                    if faiss_idx == -1:
                        continue
                    dist = distances[0][i]
                    sim_score = max(0, 1 - dist / 10)  # Convert L2 distance to similarity
                    original_idx = original_indices_map.get(faiss_idx)
                    if original_idx is not None:
                        results[original_idx] = {'semantic_score': sim_score, 'tfidf_score': 0}
        except Exception as e:
            print(f"Error during semantic search: {e}")

    # TF-IDF search
    if tfidf_vectorizer and tfidf_matrix is not None and (1 - semantic_weight) > 0:
        try:
            processed_query = preprocess_text(query)
            query_vector = tfidf_vectorizer.transform([processed_query])
            tfidf_scores = tfidf_matrix.dot(query_vector.T).toarray().flatten()

            for idx, score in enumerate(tfidf_scores):
                if score > 0.01:  # Threshold
                    if idx in results:
                        results[idx]['tfidf_score'] = score
                    else:
                        if semantic_weight == 0 or idx not in results:
                            results[idx] = {'semantic_score': 0, 'tfidf_score': score}
        except Exception as e:
            print(f"Error during TF-IDF search: {e}")

    # Combine scores
    combined_scores = {}
    for idx, scores in results.items():
        combined_score = (semantic_weight * scores['semantic_score']) + \
                         ((1 - semantic_weight) * scores['tfidf_score'])
        combined_scores[idx] = combined_score

    # Sort and return top results
    sorted_indices = sorted(combined_scores.keys(), key=lambda idx: combined_scores[idx], reverse=True)
    top_results = []

    for idx in sorted_indices[:k]:
        if idx < len(chunks):
            chunk = chunks[idx].copy()
            chunk['hybrid_score'] = combined_scores[idx]
            chunk['semantic_score'] = results[idx]['semantic_score']
            chunk['tfidf_score'] = results[idx]['tfidf_score']
            top_results.append(chunk)

    return top_results

In [ ]:
def find_relevant_figures(top_chunks, figures_tables, query, text_embedding_model, top_k=3):
    """Find relevant figures/tables based on mentions and semantic similarity"""
    if not figures_tables:
        return []

    figure_scores = {}

    # Check for explicit mentions in top chunks
    for chunk in top_chunks:
        mentioned_refs = chunk.get('figures_mentioned', [])
        for item_idx, item in enumerate(figures_tables):
            item_ref = f"{item['type']} {item['number_str']}"
            if any(item_ref.lower() == m.lower() for m in mentioned_refs):
                current_score = figure_scores.get(item_idx, 0)
                figure_scores[item_idx] = max(current_score, 5.0)  # High score for direct mention

    # Check page proximity
    chunk_pages = {chunk['page_num'] for chunk in top_chunks}
    for item_idx, item in enumerate(figures_tables):
        if item['page_num'] in chunk_pages:
            current_score = figure_scores.get(item_idx, 0)
            figure_scores[item_idx] = max(current_score, 2.0)  # Medium score for same page
        elif item['page_num']-1 in chunk_pages or item['page_num']+1 in chunk_pages:
            current_score = figure_scores.get(item_idx, 0)
            figure_scores[item_idx] = max(current_score, 0.5)  # Lower score for adjacent page

    # Check semantic similarity
    if query and text_embedding_model:
        try:
            query_embedding = text_embedding_model.encode([query])[0]
            query_tensor = torch.tensor([query_embedding])

            for item_idx, item in enumerate(figures_tables):
                desc_embedding = item.get('description_embedding')
                if desc_embedding is not None and isinstance(desc_embedding, np.ndarray):
                    desc_tensor = torch.tensor([desc_embedding])
                    similarity = util.cos_sim(query_tensor, desc_tensor).item()
                    if similarity > 0.4:  # Threshold
                        current_score = figure_scores.get(item_idx, 0)
                        figure_scores[item_idx] = max(current_score, current_score + similarity * 3.0)
        except Exception as e:
            print(f"Error during figure similarity calculation: {e}")

    # Sort figures by score
    sorted_items = sorted(figure_scores.items(), key=lambda item: item[1], reverse=True)

    # Return top items
    relevant_items = []
    seen_refs = set()

    for item_idx, score in sorted_items:
        item = figures_tables[item_idx].copy()
        item_ref = f"{item['type']}_{item['number_str']}"
        if item_ref not in seen_refs:
            item['relevance_score'] = score
            relevant_items.append(item)
            seen_refs.add(item_ref)
            if len(relevant_items) >= top_k:
                break

    return relevant_items

In [ ]:
# Fix generator function
def generate_response(query, relevant_chunks, relevant_figures_tables, generator=None, max_context_tokens=800):
    """Generate response with improved prompt and error handling"""
    if not relevant_chunks:
        return "Could not find relevant information to answer the question."

    # Prepare shorter context from chunks - use fewer chunks
    context_text = ""
    max_chunks_for_prompt = 2  # Reduced from 3
    chunks_for_prompt = relevant_chunks[:max_chunks_for_prompt]

    for chunk in chunks_for_prompt:
        # Take only first 200 words from each chunk
        chunk_text = ' '.join(chunk['text'].split()[:200])
        chunk_page = chunk['page_num'] + 1
        context_text += chunk_text + f" (Page {chunk_page})\n\n"

    # Simplified figure descriptions
    figure_descriptions = ""
    if relevant_figures_tables:
        figure_descriptions = "Relevant Figures/Tables:\n"
        for item in relevant_figures_tables[:1]:  # Only use top figure
            item_type = item['type'].capitalize()
            item_num_str = item['number_str']
            page_num = item['page_num'] + 1
            figure_descriptions += f"- {item_type} {item_num_str} (Page {page_num})\n"

    # Generate response using LLM with simplified prompt
    if generator and context_text:
        prompt = f"""Answer this question based on the provided context:
Question: {query}

Context:
{context_text.strip()}

{figure_descriptions.strip()}

Answer:"""

        try:
            # Only use max_new_tokens to avoid warning
            result = generator(prompt, max_new_tokens=150, num_return_sequences=1)[0]['generated_text']

            # Better validation
            if result and len(result.split()) > 5 and "based on the provided context" not in result.lower():
                return result.strip()
            else:
                print("Generator returned potentially invalid answer. Using fallback.")
        except Exception as e:
            print(f"Error using generator: {e}")

    # Fallback: Simpler fallback that just takes the most relevant chunk
    print("Using fallback response generation.")
    if relevant_chunks:
        fallback_answer = relevant_chunks[0]['text'][:300] + f" (Page {relevant_chunks[0]['page_num']+1})"
    else:
        fallback_answer = "Could not find relevant information."

    # Add figure reference if available
    if relevant_figures_tables:
        item = relevant_figures_tables[0]
        fallback_answer += f" (See {item['type'].capitalize()} {item['number_str']} on Page {item['page_num']+1})"

    return fallback_answer

In [ ]:
def main():
    """Optimized main execution of the MultiModal RAG pipeline"""
    print("--- Starting Optimized MultiModal RAG Pipeline ---")

    # 1. Load questions
    try:
        questions_df = pd.read_csv(QUESTIONS_FILE)
        print(f"Loaded {len(questions_df)} questions")
    except Exception as e:
        print(f"Error loading questions: {e}")
        # Create sample questions if file not found
        questions_df = pd.DataFrame({
            'ID': range(1, 12),
            'Question': [
                "What sparked the global economic crisis around 2008?",
                        "Why should we worry about unemployment rates going up?",
                        "How do economists measure economic growth without price changes messing it up?",
                        "How bad did the world economy get hit during the 2009 recession?",
                        "What happened to U.S. unemployment after the 2008 crisis kicked in?",
                        "How much did China's economy grow yearly before and during the crisis?",
                        "Did the 2008 crisis tank stock markets everywhere, or just in the U.S.?",
                        "Does fast economic growth always mean fewer people out of work in the U.S.?",
                        "Are consumer prices and overall economic prices always in sync in the U.S.?",
                        "Why has Europe struggled with jobs even before the crisis, and how bad did it get after?",
                        "How did China keep growing strong through the 2008 mess?"
            ]
        })

    # 2. Check document path
    if not os.path.exists(DOCUMENT_FILE):
        print(f"ERROR: Document not found at '{DOCUMENT_FILE}'")
        return

    # 3. Initialize models with lower complexity options
    print("\n--- Initializing Models ---")
    # Use smaller models if possible
    reranker_model, reranker_tokenizer, reranker_device = init_reranker()
    generator = init_generator()
    caption_model, feature_extractor, tokenizer, caption_device = init_image_caption_model()

    # 4. Process PDF with lower DPI
    print("\n--- Processing PDF Document ---")
    text_chunks_raw, figures_tables = extract_text_and_figures(
        DOCUMENT_FILE, caption_model, feature_extractor, tokenizer, caption_device, dpi=100
    )
    # 5. Process text chunks
    print("\n--- Processing Text Chunks ---")
    processed_chunks = process_text_chunks(text_chunks_raw, figures_tables)
    print(f"Created {len(processed_chunks)} text chunks.")
    if not processed_chunks:
        print("Error: Failed to process text chunks.")
        return

    # 6. Build retrieval indexes
    print("\n--- Building Retrieval Indexes ---")
    tfidf_vectorizer, tfidf_matrix = create_tfidf_vectorizer(processed_chunks)
    print("TF-IDF index created.")

    chunks_with_embeddings, text_embedding_model = create_text_embeddings(processed_chunks)
    if text_embedding_model is None:
        print("Warning: Text embedding model failed to load. Using TF-IDF only.")

    faiss_index = build_faiss_index(chunks_with_embeddings)
    if faiss_index is None and text_embedding_model is not None:
        print("Warning: FAISS index build failed.")

    # 7. Process questions and generate answers
    print("\n--- Answering Questions ---")
    results = []

    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df), desc="Answering Questions"):
        question_id = row['ID']
        query = row['Question']
        print(f"\nProcessing Q{question_id}: {query}")

        # Retrieve relevant chunks
        retrieved_chunks = hybrid_retrieve(
            query, chunks_with_embeddings, tfidf_vectorizer, tfidf_matrix,
            faiss_index, text_embedding_model, k=30
        )
        print(f"Retrieved {len(retrieved_chunks)} initial chunks.")

        if not retrieved_chunks:
            print("Warning: No chunks retrieved.")
            results.append({'ID': question_id, 'Text': "Retrieval failed to find relevant text.", 'Image': 0})
            continue

        # Rerank chunks
        if reranker_model and reranker_tokenizer and reranker_device:
            reranked_chunks = rerank(
                query, retrieved_chunks, reranker_model, reranker_tokenizer, reranker_device, top_k=8
            )
            print(f"Reranked to {len(reranked_chunks)} chunks.")
            if not reranked_chunks:
                print("Warning: Reranking resulted in zero chunks. Using initial retrieval as fallback.")
                reranked_chunks = retrieved_chunks[:8]
        else:
            # Skip reranking if model not loaded
            reranked_chunks = retrieved_chunks[:8]
            print("Skipped reranking (model not available).")

        # Find relevant figures/tables
        relevant_items = find_relevant_figures(
            reranked_chunks, figures_tables, query, text_embedding_model, top_k=3
        )

        if relevant_items:
            print(f"Found {len(relevant_items)} relevant figures/tables:")
            for item in relevant_items:
                print(f"  - {item['type']} {item['number_str']} (Page {item['page_num']+1}), "
                      f"Score: {item.get('relevance_score', 'N/A'):.2f}")
        else:
            print("No relevant figures/tables found for this query.")

        # Generate response
        response_text = generate_response(
            query, reranked_chunks, relevant_items, generator
        )
        print(f"Generated Response: {response_text[:100]}...")

        # Determine most relevant figure number for submission
        figure_num_output = 0  # Default
        if relevant_items:
            try:
                # Extract number after hyphen
                num_part = relevant_items[0]['number_str'].split('-')[-1]
                figure_num_output = int(num_part)
            except:
                print(f"Warning: Could not parse figure number from '{relevant_items[0]['number_str']}'. Using 0.")
                figure_num_output = 0

        results.append({
            'ID': question_id,
            'Text': response_text,
            'Image': figure_num_output
        })

    # 8. Save results
    print("\n--- Saving Results ---")
    results_df = pd.DataFrame(results)
    submission_path = 'submission.csv'

    try:
        results_df.to_csv(submission_path, index=False)
        print(f"Submission saved successfully to {submission_path}")
    except Exception as e:
        print(f"Error saving submission file: {e}")

    # 9. Display preview
    print("\nSubmission Preview:")
    print(results_df.head())

    print("\n--- Pipeline Finished ---")
    return results_df

In [ ]:
# Run the pipeline
if __name__ == "__main__":
    main()

--- Starting Optimized MultiModal RAG Pipeline ---
Loaded 11 questions

--- Initializing Models ---
Loading reranker model: BAAI/bge-reranker-base...


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

...

Generated Response: To prevent workers from losing their jobs, they make it expensive for firms to lay off workers...

Processing Q11: How did China keep growing strong through the 2008 mess?
Retrieved 30 initial chunks.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=150) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reranked to 8 chunks.
Found 3 relevant figures/tables:
  - table 1-4 (Page 10), Score: 6.95
  - figure 1-6 (Page 10), Score: 6.83
  - table 1-1 (Page 2), Score: 6.44


Answering Questions: 100%|██████████| 11/11 [00:07<00:00,  1.45it/s]

Generated Response: rules were devised and implemented to contain government spending...

--- Saving Results ---
Submission saved successfully to submission.csv

Submission Preview:
   ID                                               Text  Image
0   1  for most of the decade led to strong growth of...      1
1   2  higher unemployment rate is typically associat...      5
2   3  is the sum of the quantities of final goods ti...      1
3   4                            3.2 1.5 2.3 4.0 3.0 3.2      1
4   5  Unemployment increased dramati- cally, to near...      2

--- Pipeline Finished ---
